# Section 1: System prompt
This notebook evaluates different system prompts for the QueryChat assistant used in our AI Burnout Dashboard. In this section we aim to determine which prompt best helps the LLM understand the dataset and provide useful insights to users.

### Test questions
1. **Which job roles should HR prioritize for burnout interventions?**  
   Tests whether the model goes beyond ranking numbers and frames findings as a prioritization problem.

2. **Does high AI usage help or hurt employee wellbeing?**  
   Tests whether the model distinguishes productivity gains from wellbeing trade-off, which might be a HR concern.

3. **Are high-productivity employees at greater burnout risk?**  
   Tests whether the model can reason about productivity-burnout tension and give actionable suggestions.

The respond from AI explorer would follow the corresponding question, eliminating the SQL query code.

## Experiments

### 1. Load data

In [1]:
import pandas as pd
import anthropic
import os
import sys
from IPython.display import display, Markdown
from querychat import QueryChat
from chatlas import ChatAnthropic
from dotenv import load_dotenv
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.data import load_dashboard_data

load_dotenv()

df = load_dashboard_data()

client = anthropic.Anthropic()

In [ ]:
# Define prompts and questions for experimentation
PROMPT_1 = "Hi! Ask questions about the dataset."

PROMPT_2 = """You are a data assistant. The dataset contains employee-level workplace wellbeing and productivity data.

Columns include: job_role, experience_years, ai_tool_usage_hours_per_week, tasks_automated_percent,
manual_work_hours_per_week, meeting_hours_per_week, collaboration_hours_per_week, focus_hours_per_day,
deadline_pressure_level, burnout_risk_score, burnout_risk_level, productivity_score,
work_life_balance_score, workload_score, workload_band, ai_band.

Answer questions about this dataset accurately."""

PROMPT_3 = """You are a workplace analytics assistant helping HR leaders explore employee burnout risk.

Context:
This dashboard analyzes how AI usage, workload, and work habits relate to employee burnout and productivity.

Target users:
HR leaders and people analytics professionals who want to make evidence-based decisions about AI adoption
and workforce wellbeing.

Your role:
- Help users identify patterns in burnout risk, productivity, and AI usage.
- Explain how workplace factors may relate to employee burnout.
- Provide insights that support responsible AI deployment and sustainable productivity.

Important analysis goals:
1. Distinguish burnout driven by workload from burnout potentially associated with AI usage.
2. Compare burnout risk across job roles and experience levels.
3. Identify whether productivity gains occur alongside increased burnout risk.

Guidelines:
- Use clear, non-technical language suitable for HR managers.
- Highlight actionable insights rather than only describing statistics.
- Suggest useful follow-up questions that help users explore the dataset further.
- If a question refers to a variable not in the dataset (e.g., salary), clarify that this dataset
  focuses on wellbeing and productivity rather than compensation."""

prompts = {
    "Prompt 1 (Minimal)": PROMPT_1,
    "Prompt 2 (Schema context)": PROMPT_2,
    "Prompt 3 (analytics assistant)": PROMPT_3,
}

questions = [
    "Which job roles should HR prioritize for burnout interventions?",
    "Does high AI usage help or hurt employee wellbeing?",
    "Are high-productivity employees at greater burnout risk?",
]


In [3]:
# Precalculated data for LLM

# Q1
context_q1 = df.groupby("job_role").agg(
    avg_burnout=("burnout_risk_score", "mean"),
    high_burnout_pct=("burnout_risk_level", lambda x: (x=="High").mean() * 100),
    avg_manual_hours=("manual_work_hours_per_week", "mean"),
    avg_deadline_pressure=("workload_score", "mean"),
    n=("Employee_ID", "count")
).round(2).sort_values("avg_burnout", ascending=False).to_string()

# Q2
context_q2 = df.groupby("ai_band", observed=True).agg(
    avg_burnout=("burnout_risk_score", "mean"),
    avg_productivity=("productivity_score", "mean"),
    avg_wlb=("work_life_balance_score", "mean"),
    avg_ai_hours=("ai_tool_usage_hours_per_week", "mean"),
    n=("Employee_ID", "count")
).round(2).to_string()

# Q3
df["productivity_band"] = pd.qcut(df["productivity_score"], q=3, labels=["Low", "Medium", "High"])
context_q3 = df.groupby("productivity_band", observed=True).agg(
    avg_burnout=("burnout_risk_score", "mean"),
    high_burnout_pct=("burnout_risk_level", lambda x: (x=="High").mean() * 100),
    avg_wlb=("work_life_balance_score", "mean"),
    n=("Employee_ID", "count")
).round(2).to_string()

question_contexts = {
    questions[0]: context_q1,
    questions[1]: context_q2,
    questions[2]: context_q3,
}

In [4]:
# Define functions to display responses
def ask_with_prompt(system_prompt, question, df):
    context = question_contexts[question]

    user_message = f"""{question}

Here is the relevant data summary for this question:
{context}
"""
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=600,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

def display_response(prompt_label: str, question: str, response: str):
    display(Markdown(f"### {prompt_label}\n**Q:** {question}\n\n**Response:**\n\n{response}"))
    print("-" * 80)

## Question 1 — Which job roles should HR prioritize for burnout interventions?

In [5]:
q1 = questions[0]
responses_q1 = {}

for label, prompt in prompts.items():
    response = ask_with_prompt(prompt, q1, df)
    responses_q1[label] = response
    display_response(label, q1, response)

### Prompt 1 (Minimal)
**Q:** Which job roles should HR prioritize for burnout interventions?

**Response:**

Based on the data analysis, HR should prioritize burnout interventions in the following order:

## **Top Priority: Managers**
- **Highest burnout rates**: 8.95 average burnout score and 86.2% experiencing high burnout
- **Key factors**: Highest deadline pressure (43.05) and significant manual work hours (26.19)
- **Impact**: With 652 managers affected, this represents critical leadership burnout that can cascade throughout the organization

## **Second Priority: Developers**
- **High volume concern**: 1,115 employees (largest group) with 73.7% experiencing high burnout
- **Burnout level**: 8.37 average score indicates severe stress levels
- **Strategic importance**: Developer burnout can significantly impact product delivery and technical innovation

## **Third Priority: Writers**
- **Comparable burnout**: 8.35 average score with 73.5% high burnout rate
- **Moderate group size**: 461 employees but with concerning stress levels similar to developers

## **Recommended Intervention Strategy:**
1. **Immediate focus on managers** - Address deadline pressure and provide leadership support tools
2. **Scale interventions for developers** - Given the large population size, implement systematic stress reduction programs
3. **Targeted support for writers** - Focus on workload management and creative process optimization

The data shows a clear hierarchy where managerial burnout poses the highest risk due to both severity and organizational impact, followed by volume-based concerns for developers and writers.

--------------------------------------------------------------------------------


### Prompt 2 (Schema context)
**Q:** Which job roles should HR prioritize for burnout interventions?

**Response:**

Based on the data, HR should prioritize burnout interventions for the following job roles, in order of urgency:

## 1. **Managers** (Highest Priority)
- **86.2%** have high burnout risk (highest across all roles)
- Average burnout score of **8.95** (highest)
- Highest deadline pressure levels (**43.05**)
- High manual work hours (**26.19** per week)

## 2. **Developers** (Second Priority)
- **73.7%** have high burnout risk
- Second highest burnout score (**8.37**)
- Largest group size (1,115 employees) - high impact potential

## 3. **Writers** (Third Priority)
- **73.5%** have high burnout risk
- High burnout score (**8.35**)
- Despite lower deadline pressure, still showing significant burnout

## Key Insights:
- **Managers** stand out with dramatically higher burnout rates and pressure levels
- The top three roles all have **70%+** of employees at high burnout risk
- **Analysts** and **Designers** have somewhat lower but still concerning burnout rates (69.8% and 66.6% respectively)

## Recommended Approach:
1. **Immediate intervention** for Managers - focus on workload management and deadline pressure reduction
2. **Scaled intervention** for Developers given the large population size
3. **Targeted support** for Writers to understand unique stressors despite lower pressure levels

The data suggests a clear burnout crisis, particularly in management roles, requiring urgent organizational attention.

--------------------------------------------------------------------------------


### Prompt 3 (HR analytics assistant)
**Q:** Which job roles should HR prioritize for burnout interventions?

**Response:**

Based on this analysis, HR should prioritize burnout interventions in this order:

## **Immediate Priority: Managers**
- **Highest risk**: 86% experience high burnout (8.95/10 average)
- **Key insight**: Despite having fewer manual work hours (26), managers face the highest deadline pressure (43.05)
- **Root cause likely**: Leadership responsibilities, decision-making stress, and being accountable for team outcomes

## **Secondary Priority: Developers and Writers**
- **Similar risk levels**: ~74% high burnout rate for both roles
- **Different patterns**: 
  - Developers have lower manual hours (19) but moderate deadline pressure
  - Writers have similar manual workload but slightly higher deadline pressure
- **Intervention focus**: Role-specific stress management and workload optimization

## **Monitoring Priority: Marketers and Analysts**
- **Moderate risk**: 69-72% high burnout rates
- **Balanced workloads**: More manual hours than developers but manageable deadline pressure
- **Opportunity**: These roles may benefit from preventive measures rather than crisis intervention

## **Lowest Priority: Designers**
- **Best positioned**: 67% high burnout rate (still concerning, but relatively better)
- **Highest manual hours** but **lowest deadline pressure**
- **Focus**: Maintain current wellbeing while monitoring for changes

## **Recommended Actions:**
1. **Start with management coaching** and leadership support programs
2. **Investigate why developers and writers have high burnout** despite reasonable workloads
3. **Consider whether AI tools are adding complexity** rather than reducing stress for technical roles

Would you like me to explore how AI usage patterns differ across these job roles, or analyze what specific interventions might work best for managers?

--------------------------------------------------------------------------------


## Question 2 — Does high AI usage help or hurt employee wellbeing?

In [6]:
q2 = questions[1]
responses_q2 = {}

for label, prompt in prompts.items():
    response = ask_with_prompt(prompt, q2, df)
    responses_q2[label] = response
    display_response(label, q2, response)

### Prompt 1 (Minimal)
**Q:** Does high AI usage help or hurt employee wellbeing?

**Response:**

Based on the data, high AI usage appears to have **mixed effects** on employee wellbeing:

## Positive Effects:
- **Productivity boost**: High AI users show dramatically higher productivity (73.38) compared to low users (56.92) - a 29% increase
- **Manageable burnout**: Burnout levels remain relatively stable across all AI usage groups (8.33-8.40 on the scale)

## Negative Effects:
- **Work-life balance decline**: High AI users report lower work-life balance scores (4.61) compared to low (4.75) and moderate users (4.81)

## Key Insight:
The "Moderate" AI usage group (averaging ~10 hours) appears to hit a "sweet spot" - they maintain the highest work-life balance (4.81) while still achieving strong productivity gains (64.78) compared to low users.

## Bottom Line:
High AI usage helps productivity significantly without causing severe burnout, but it does come at the cost of work-life balance. Organizations might consider promoting moderate AI adoption to maximize benefits while protecting employee wellbeing, rather than pushing for maximum AI usage.

The data suggests that while AI can be a powerful productivity tool, there may be diminishing returns to wellbeing when usage becomes excessive.

--------------------------------------------------------------------------------


### Prompt 2 (Schema context)
**Q:** Does high AI usage help or hurt employee wellbeing?

**Response:**

Based on the data, high AI usage shows a **mixed impact** on employee wellbeing:

## Positive Effects:
- **Significantly higher productivity**: Employees with high AI usage score 73.38 vs. 56.92 for low usage (+29% increase)
- **Minimal burnout increase**: Burnout scores are nearly identical across all AI usage levels (8.33-8.40)

## Negative Effects:
- **Slightly lower work-life balance**: High AI users score 4.61 vs. 4.75 for low users (-3% decrease)

## Key Takeaway:
High AI usage appears to be **primarily beneficial** for employees. While there's a small dip in work-life balance, the substantial productivity gains come with virtually no increase in burnout risk. This suggests that AI tools are effectively helping employees work more efficiently without significantly harming their wellbeing.

The productivity boost (+29%) far outweighs the modest work-life balance decline (-3%), indicating that when implemented well, AI tools can enhance performance while maintaining sustainable stress levels.

--------------------------------------------------------------------------------


### Prompt 3 (HR analytics assistant)
**Q:** Does high AI usage help or hurt employee wellbeing?

**Response:**

Based on the data, the relationship between AI usage and employee wellbeing shows some nuanced patterns:

## Key Findings:

**Burnout Risk**: High AI users show only slightly higher burnout scores (8.40) compared to low (8.33) and moderate users (8.33). This difference is quite small, suggesting AI usage alone isn't a major burnout driver.

**Work-Life Balance**: There's a small decline in work-life balance as AI usage increases:
- Low AI users: 4.75/10
- Moderate AI users: 4.81/10  
- High AI users: 4.61/10

**Productivity Gains**: High AI users show significantly higher productivity (73.38) compared to low users (56.92) - nearly a 30% increase.

## What This Means for HR:

The data suggests **AI usage isn't inherently harmful to wellbeing**, but there are some trade-offs to monitor:

✅ **Positive**: Substantial productivity gains without major burnout increases
⚠️ **Watch point**: Slight decline in work-life balance among heavy AI users

## Recommended Actions:

1. **Focus on work-life balance support** for high AI users - they may be working longer hours despite efficiency gains
2. **Investigate what's driving the productivity boost** - is it genuine efficiency or extended working hours?
3. **Consider workload management** to ensure productivity gains translate to sustainable work practices

## Useful Follow-up Questions:
- How does AI usage vary by job role and experience level?
- Are high AI users working more hours despite being more productive?
- What specific workplace factors correlate most strongly with burnout in our organization?

Would you like to explore any of these areas further?

--------------------------------------------------------------------------------


## Question 3 — Are high-productivity employees at greater burnout risk?

In [7]:
q3 = questions[2]
responses_q3 = {}

for label, prompt in prompts.items():
    response = ask_with_prompt(prompt, q3, df)
    responses_q3[label] = response
    display_response(label, q3, response)

### Prompt 1 (Minimal)
**Q:** Are high-productivity employees at greater burnout risk?

**Response:**

Looking at the data, **high-productivity employees actually have lower burnout risk**, not higher.

Here are the key findings:

## Burnout decreases as productivity increases:
- **Low productivity**: 9.13 average burnout score, 89.77% high burnout rate
- **Medium productivity**: 8.40 average burnout score, 74.67% high burnout rate  
- **High productivity**: 7.53 average burnout score, 55.60% high burnout rate

## The relationship shows that:
- High-productivity employees have 34 percentage points lower high burnout rates than low-productivity employees (55.6% vs 89.8%)
- They also report significantly better work-life balance (5.57 vs 3.80)

This suggests that rather than being "burned out overachievers," high-productivity employees may be more effective at managing their workload and maintaining sustainable work practices. The strong correlation with better work-life balance indicates they've likely developed better boundaries and efficiency strategies that protect them from burnout while maintaining high performance.

--------------------------------------------------------------------------------


### Prompt 2 (Schema context)
**Q:** Are high-productivity employees at greater burnout risk?

**Response:**

Based on the data, **high-productivity employees are actually at lower burnout risk**, not higher risk.

Here are the key findings:

**Burnout Risk decreases with higher productivity:**
- Low productivity employees: 9.13 average burnout score, 89.77% at high burnout risk
- Medium productivity employees: 8.40 average burnout score, 74.67% at high burnout risk  
- High productivity employees: 7.53 average burnout score, 55.60% at high burnout risk

**Work-life balance improves with higher productivity:**
- Low productivity: 3.80 average work-life balance score
- Medium productivity: 4.82 average work-life balance score
- High productivity: 5.57 average work-life balance score

This suggests that high-productivity employees have developed better work practices, time management, or work environments that allow them to be both more productive and maintain better wellbeing. The relationship appears to be inverse - as productivity increases, burnout risk decreases and work-life balance improves.

This could indicate that productivity and burnout aren't simply a matter of working harder, but rather working more effectively and sustainably.

--------------------------------------------------------------------------------


### Prompt 3 (HR analytics assistant)
**Q:** Are high-productivity employees at greater burnout risk?

**Response:**

Based on this data, **high-productivity employees actually show lower burnout risk** compared to their lower-productivity colleagues. Here are the key insights:

## Lower Burnout Risk for High Performers
- **High-productivity employees**: 56% are at high burnout risk
- **Medium-productivity employees**: 75% are at high burnout risk  
- **Low-productivity employees**: 90% are at high burnout risk

## Better Work-Life Balance
High-productivity employees also report significantly better work-life balance (5.57 out of 10) compared to medium (4.82) and low-productivity employees (3.80).

## What This Suggests
This pattern indicates that high productivity may actually be **protective against burnout** rather than causing it. Possible explanations include:

- **Efficiency advantage**: High performers may complete work more effectively, reducing time pressure
- **Better work habits**: They might have developed sustainable practices that support both performance and wellbeing
- **Resource access**: High performers may receive better support, tools, or autonomy
- **Skill mastery**: Greater competence can reduce work-related stress

## Actionable Implications
- Study what high performers do differently - their practices might benefit struggling employees
- Focus burnout prevention efforts on low and medium performers who show the highest risk
- Consider whether low performers need additional training, resources, or workload adjustments

**Follow-up question to explore**: What specific work habits or AI usage patterns differentiate high performers from others?

--------------------------------------------------------------------------------


## Summary

### Key Observations

**Prompt 1 (Minimal):** With almost no context, the model still generates accurate responses because QueryChat understands the data well from just the SQL output. However, the model produces accurate but generic responses that simply summarize the query results, with little advice for HR decision-making.

**Prompt 2 (Schema context):** Adding column descriptions produced slightly more concise responses, but very similar to minimal prompt. The improvement was hardly seen because the model already infers schema from the data. The main effect was reducing verbosity, not improving insight quality.

Because the SQL generation ability is largely handled by QueryChat internally, the main benefit of prompt customization lies in improving the interpretability and usefulness of the LLM’s responses for dashboard users. We also decided to remove data description as AI explorer seems to understand data well without context. With these observations, we designed prompt 3.

**Prompt 3 (HR analytics assistant):** Establishing a target audience (HR leaders) and analysis goals (workload vs AI-driven burnout, productivity trade-offs) produced the most useful responses for target users, who might not necessarily be statisticians. The model framed findings as decision-relevant insights and suggested actionable follow-up questions. 

### Prompt Design Decision

We adopted **Prompt 3** for the AI Explorer tab. The key design choices were:
- Specifying the target user (HR leaders, not data scientists) shifts the model toward plain-language explanations
- Listing explicit analysis goals (workload vs AI burnout, productivity trade-offs) guides the model to explore the most relevant patterns for this dashboard
- Omitting a full column-by-column data description, since QueryChat already has schema awareness. The prompt focuses on *how* to interpret results rather than *what* columns exist

These choices were validated by comparing the same three questions across all prompts using the clarity, relevance, and actionability.